In [1]:
# ===========================================================================
# UA-SPEECH MODEL TRAINING - interactive driver
#
# All training logic lives in src/training/; this notebook only calls it and
# stores results, matching notebooks/01/02's convention - functions and
# architecture belong in src/, only the act of running training and storing
# models happens here.
#
#   src/training/models.py     model factory: acoustic / deep_frozen / deep_lora / fusion
#   src/training/runner.py     TrainingConfig, run_training() - the fold loop
#   src/training/baseline.py   Phase 2: frozen wav2vec + linear SVM baseline
#   src/training/engine.py     one epoch: AMP, gradient clipping, optimizer
#   src/training/reporting.py  predictions / metrics / confusion-matrix / ROC / embeddings I/O
#
# Every run below is a TrainingConfig + run_training() pair - the "experiment
# manager" IS TrainingConfig (src/training/runner.py): every
# checkpoint/metric/prediction/confusion-matrix/ROC/embedding already lands
# under outputs/<kind>/<run_name>/, keyed by run_name, so adding a future
# ablation means appending a TrainingConfig to a family list below, not
# writing new plumbing.
#
# See ROADMAP.md for the phase plan this notebook implements (Phase 1 sanity
# check, Phase 2 baseline reproduction, Phase 3 ablation).
# ===========================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("UA-Speech Training Notebook")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))
print_kv("Speakers", df_m6["Speaker_ID"].nunique())


══════════════════════════════════════════════════════════════════════════════
  UA-SPEECH TRAINING NOTEBOOK
══════════════════════════════════════════════════════════════════════════════
  Manifest ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv
  Utterances .............................. 21381
  Speakers ................................ 28


In [2]:
# EXPERIMENT MANAGER - the three pathway families every ablation variant
# (src.training.models.MODEL_NAMES) belongs to. Grouping by family, rather
# than looping over all six variants in one cell, is what lets MFCC-only,
# Wav2Vec2-only, and Fusion training be run, resumed, or extended
# independently in the stages below - a future ablation variant is added by
# appending its model name to the right family list here, nothing else
# changes.
MFCC_FAMILY = ["acoustic"]
WAV2VEC_FAMILY = ["deep_frozen", "deep_lora"]
FUSION_FAMILY = ["fusion_frozen", "fusion", "attention_fusion", "attention_fusion_praat"]

# PRIMARY_DETECTION_MODELS - the supervisor-requested six-variant budgeted
# sweep (Stage 8d below): exactly the variants needed to answer "does LoRA
# beat frozen wav2vec2?" (deep_frozen vs deep_lora), "does LoRA help fusion?"
# (fusion_frozen vs fusion), and "what does the proposed architecture buy?"
# (attention_fusion vs everything else) - no attention_fusion_praat or
# severity task here, matching "only the necessary variants" (Phase 3
# already spends the bulk of GPU time on 6-variant x 28-fold LOSO without
# also motivating the 3rd Praat pathway or the severity task in the same
# 6-hour budget).
PRIMARY_DETECTION_MODELS = ["acoustic", "deep_frozen", "deep_lora",
                           "fusion_frozen", "fusion", "attention_fusion"]

print_header("Experiment Manager - Pathway Families")
print_kv("MFCC-only", ", ".join(MFCC_FAMILY))
print_kv("Wav2Vec2-only", ", ".join(WAV2VEC_FAMILY))
print_kv("Fusion", ", ".join(FUSION_FAMILY))
print_kv("Primary detection sweep (Stage 8d)", ", ".join(PRIMARY_DETECTION_MODELS))


══════════════════════════════════════════════════════════════════════════════
  EXPERIMENT MANAGER - PATHWAY FAMILIES
══════════════════════════════════════════════════════════════════════════════
  MFCC-only ............................... acoustic
  Wav2Vec2-only ........................... deep_frozen, deep_lora
  Fusion .................................. fusion_frozen, fusion, attention_fusion, attention_fusion_praat
  Primary detection sweep (Stage 8d) ...... acoustic, deep_frozen, deep_lora, fusion_frozen, fusion, attention_fusion


In [ ]:
# EXECUTION MODE — set this before running anything below.
#
# The pre-repair notebook had one mode: launch everything and hope it fits.
# It did not fit. The 2.5h primary sweep spent 2.58h on three of six variants
# and silently dropped the other three — including the proposed attention-
# fusion architecture — then wrote a comparison table from the three that
# finished. Every detection row in it was a single held-out healthy-control
# speaker, so every class-sensitive metric in it was undefined.
#
# Three explicit modes now, with a pre-flight cost projection before any
# expensive stage:
#
#   SMOKE        2 folds, few epochs, sample-capped. Answers "does everything
#                train, checkpoint, predict and score?" Never a result.
#   DEVELOPMENT  every variant on ONE shared class-balanced protocol
#                (screening folds — each holds out both control AND
#                dysarthric speakers, so per-fold metrics are defined and
#                folds are independent units for Wilcoxon/bootstrap).
#                Answers "which architectures are promising?"
#   FINAL        the full intended LOSO protocol. Only after DEVELOPMENT has
#                ranked the variants, and only with the wall-clock to finish.
#
# MEASURED COST (outputs/metrics/budget_manager_*_log.json, this GPU):
#   full 28-fold LOSO, all seven variants   ~119 GPU-hours
#   screening-8,       all seven variants    ~34 GPU-hours
# Neither fits one session. Both resume across sessions without recomputation
# — run_training loads completed folds from disk (see _load_completed_fold).
from src.console import print_note, print_status
MODE = "SMOKE"          # "SMOKE" | "DEVELOPMENT" | "FINAL"

MODE_SETTINGS = {
    "SMOKE":       dict(cv_protocol="screening", screening_folds=8, max_folds=2,
                        epochs=3, patience=2, limit_samples=400, hard_cap_hours=1.0),
    "DEVELOPMENT": dict(cv_protocol="screening", screening_folds=8, max_folds=None,
                        epochs=10, patience=3, limit_samples=None, hard_cap_hours=6.0),
    "FINAL":       dict(cv_protocol="loso", screening_folds=8, max_folds=None,
                        epochs=config.DEFAULT_EPOCHS, patience=config.DEFAULT_PATIENCE,
                        limit_samples=None, hard_cap_hours=6.0),
}
if MODE not in MODE_SETTINGS:
    raise ValueError(f"MODE must be one of {list(MODE_SETTINGS)}, got {MODE!r}")

SETTINGS = MODE_SETTINGS[MODE]
# Prefixing by mode keeps the three kinds of run separately addressable on
# disk. '_smoke_' also matches src.results.NON_EXPERIMENT_PREFIXES, so smoke
# runs are structurally excluded from every report rather than relying on
# anyone remembering they were throwaway.
RUN_PREFIX = {"SMOKE": "_smoke_", "DEVELOPMENT": "dev_", "FINAL": "final_"}[MODE]

# Folds each variant will actually be evaluated on — the denominator the
# registry records as expected_folds, and the multiplier in the cost
# projection below.
EXPECTED_FOLDS = (SETTINGS["max_folds"] if SETTINGS["max_folds"] is not None
                  else (SETTINGS["screening_folds"] if SETTINGS["cv_protocol"] == "screening"
                        else len(config.ALL_SPEAKERS)))

print_header(f"Execution mode — {MODE}")
print_kv("Protocol", SETTINGS["cv_protocol"])
print_kv("Folds per variant", EXPECTED_FOLDS)
print_kv("Epochs / patience", f"{SETTINGS['epochs']} / {SETTINGS['patience']}")
print_kv("Sample cap", SETTINGS["limit_samples"] or "none (full split)")
print_kv("Run-name prefix", RUN_PREFIX)
print_kv("Session hard cap", f"{SETTINGS['hard_cap_hours']}h")

if MODE == "SMOKE":
    print_note("SMOKE mode — nothing produced here is a scientific result. Runs are "
               "prefixed '_smoke_' so src.results excludes them from every report.")
elif MODE == "FINAL":
    print_note("FINAL mode — run the budget pre-flight before launching. At measured "
               "cost a full LOSO sweep of all seven variants is ~119 GPU-hours and "
               "will NOT finish in one session. Plan to resume across sessions, or "
               "reduce scope deliberately, rather than discovering a truncated "
               "leaderboard afterwards.")

# Detection folds must contain BOTH classes for their metrics to be defined.
# Screening folds are built class-stratified (src.splits.build_screening_folds);
# LOSO folds are single-class by construction but pool to both across the
# sweep — which is why the interleaved config.ALL_SPEAKERS order matters for
# any run that stops early.
if SETTINGS["cv_protocol"] == "screening":
    from src.splits import build_screening_folds

    _controls = set(config.CONTROL_IDS)
    _folds = build_screening_folds(SETTINGS["screening_folds"], config.DEFAULT_SEED)
    _evaluated = _folds[:EXPECTED_FOLDS]
    _single_class = [i for i, fold in enumerate(_evaluated, start=1)
                     if not (0 < sum(s in _controls for s in fold) < len(fold))]
    print_status(
        f"All {len(_evaluated)} evaluated screening fold(s) hold out both classes"
        if not _single_class
        else f"Single-class screening fold(s) — metrics will be undefined: {_single_class}",
        ok=not _single_class)
else:
    print_note(f"LOSO: each of the {EXPECTED_FOLDS} folds holds out ONE speaker and is "
               "single-class by construction, so per-fold precision/recall/F1/AUROC "
               "are undefined (NaN). Only the POOLED metrics are reportable. "
               "config.ALL_SPEAKERS is interleaved, so a partial run still covers "
               "both classes.")


In [3]:
# STAGE 1 - Pipeline sanity check ("smoke test"). Trains the cheapest model
# (MFCC-only) for one fold, one epoch, on a tiny slice of data. This is NOT a
# real result - it exists to confirm the whole chain (model init, optimizer,
# scheduler, AMP, gradient clipping, early stopping, checkpointing,
# TensorBoard logging, prediction/metric/confusion-matrix/ROC/embedding
# writers) actually runs end to end before spending GPU time on a real run.
from src.training.runner import TrainingConfig, run_training

smoke_cfg = TrainingConfig(
    task="detection", model="acoustic",
    epochs=1, max_folds=1, limit_samples=24,
    run_name="_smoke_test",
)
run_training(df_m6, smoke_cfg)


╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│               Model A — MFCC 1D-CNN (cepstral features only)               │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ................................... acoustic — Model A — MFCC 1D-CNN (cepstral features only)
  Cross-validation protocol ............... Leave-One-Speaker-Out
  Run name ................................ _smoke_test
  Device .................................. cuda
  Epochs / batch size ..................... 1 / 32
  LR (head / wav2vec backbone) ............ 0.001 / 0.0001
  Early stopping patience ................. 5 epochs on validation loss

─── Front end — short-time analysis ──────────────────────────────────────────
  Sa

(                       mean  std
 test_loss          0.660319  NaN
 accuracy           0.750000  NaN
 precision          0.000000  NaN
 recall             0.000000  NaN
 specificity        0.750000  NaN
 f1                 0.000000  NaN
 auroc                   NaN  NaN
 fold_time_s       15.547000  NaN
 train_time_s      10.391000  NaN
 inference_time_s   5.156000  NaN,
 {'accuracy': 0.75,
  'precision': 0.0,
  'recall': 0.0,
  'specificity': 0.75,
  'f1': 0.0,
  'auroc': nan})

In [4]:
# STAGE 2 - Phase 2, step 1: reproduce the ICASSP base paper's feature
# extractor. Frozen wav2vec 2.0 (no LoRA, no fine-tuning) -> one 768-dim
# embedding per utterance PER HIDDEN-STATE LAYER (13 layers: the CNN
# feature-extractor output + 12 transformer layers). The base paper finds
# different layers win for different tasks (layer 1 for detection, layer 13
# for severity), so all 13 are extracted here rather than just the final
# layer - Stage 3 sweeps them to find which wins on this reproduction.
# Identical across every LOSO fold, so extracted once and cached to
# outputs/embeddings/.
from src.training.baseline import extract_frozen_embeddings_all_layers

frozen_embeddings_all_layers = extract_frozen_embeddings_all_layers(df_m6, batch_size=16)
print(f"Frozen embeddings (all layers): {frozen_embeddings_all_layers.shape}")

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 11811.14it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



══════════════════════════════════════════════════════════════════════════════
  EXTRACTING FROZEN WAV2VEC 2.0 EMBEDDINGS (ALL 13 LAYERS)
══════════════════════════════════════════════════════════════════════════════
  Utterances .............................. 21381
  Device .................................. cuda
  Frozen wav2vec 2.0 forward (13 layers) 100%|███████████████| 1337/1337 [04:26<00:00,  5.02batch/s]
  Frozen per-layer embeddings ............. extracted and cached to C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\embeddings\frozen_wav2vec_all_layers.npz
Frozen embeddings (all layers): (21381, 13, 768)


In [5]:
# STAGE 3 - Phase 2, step 2: frozen wav2vec 2.0 -> linear SVM, swept across
# all 13 layers and evaluated on the full 28-fold LOSO detection protocol -
# exactly the base paper's pipeline and per-layer comparison. The paper's
# own reported result is layer 1 at 93.95% accuracy; the best layer found
# here is the number every other model in this project has to beat to be a
# genuine improvement, not an assumed one. If the best layer or accuracy
# lands far from the paper's, that's worth investigating (preprocessing,
# VAD, clip length) before trusting the ablation table in Stage 8.
from src.training.baseline import sweep_svm_baseline_layers

detection_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="detection", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

best_layer = int(detection_layer_sweep.iloc[0]["layer"])
baseline_pooled = detection_layer_sweep.iloc[0].to_dict()
baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Detection")
print_kv("Best layer", f"{best_layer} (paper reports layer 1 at 93.95% accuracy)")
print_kv("Accuracy", f"{baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{baseline_pooled['f1']:.4f}")
print_kv("Recall (sensitivity)", f"{baseline_pooled['recall']:.4f}")
print_kv("Precision", f"{baseline_pooled['precision']:.4f}")
print_kv("Specificity", f"{baseline_pooled['specificity']:.4f}")
print_kv("AUROC", f"{baseline_pooled['auroc']:.4f}")


─── Layer 0 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... detection
  Run name ................................ baseline_svm_detection_layer0
  Fitting SVM per LOSO fold        100%|██████████████████████████| 28/28 [04:48<00:00, 10.32s/fold]

─── Per-fold mean +/- std ────────────────────────────────────────────────────
       metric   mean    std
     accuracy 0.7793 0.2478
    precision 0.5357 0.5079
       recall 0.3897 0.4286
  specificity 0.3896 0.4389
           f1 0.4267 0.4534
        auroc    NaN    NaN

─── Pooled across all folds (the base-paper-comparable numbers) ──────────────
  accuracy       ████████████████░░░░   0.7791 →
  precision      █████████████████░░░   0.8386  
  recall         █████████

In [6]:
# STAGE 4 - Phase 2, step 3: the same frozen wav2vec 2.0 + linear SVM layer
# sweep, on the severity task's 81-fold balanced leave-one-per-class-out
# protocol (config.DROPPED_FOR_BALANCE, corrected to match the base paper's
# stated exclusion criterion - see src/config.py). The paper's own reported
# result is layer 13 (final) at 44.56% accuracy (4-class) - a low absolute
# number, expected for a 4-way severity task, but the target to compare
# against before trusting the severity ablation in Stage 13.
from src.training.baseline import sweep_svm_baseline_layers

severity_layer_sweep = sweep_svm_baseline_layers(
    df_m6, task="severity", all_layer_embeddings=frozen_embeddings_all_layers, max_folds=None)

severity_best_layer = int(severity_layer_sweep.iloc[0]["layer"])
severity_baseline_pooled = severity_layer_sweep.iloc[0].to_dict()
severity_baseline_pooled.pop("layer")

print_header("Baseline (Frozen wav2vec 2.0 + Linear SVM) - Severity")
print_kv("Best layer", f"{severity_best_layer} (paper reports layer 13/final at 44.56% accuracy)")
print_kv("Accuracy", f"{severity_baseline_pooled['accuracy']:.4f}")
print_kv("F1", f"{severity_baseline_pooled['f1']:.4f}")


─── Layer 0 / 12 ─────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════════════
  PHASE 2 BASELINE: FROZEN WAV2VEC 2.0 + LINEAR SVM
══════════════════════════════════════════════════════════════════════════════
  Task .................................... severity
  Run name ................................ baseline_svm_severity_layer0

══════════════════════════════════════════════════════════════════════════════
  SEVERITY SPLITS (BALANCED LEAVE-ONE-PER-CLASS-OUT)
══════════════════════════════════════════════════════════════════════════════
  Speakers dropped for balance ............ ['M12', 'M08', 'M09']

─── Speakers per severity class ──────────────────────────────────────────────
  High .................................... F05, M10, M14
  Low ..................................... F02, M07, M16
  Mid ..................................... F04, M05, M11
  Very Low ................................ F03, M0

In [7]:
# STAGE 5 - MFCC-only screening (detection). Cheap 8-fold speaker-grouped
# screening run (not full 28-fold LOSO) of the Acoustic Pathway alone
# (Model A) - the cheapest of the six ablation variants, and the one Stage
# 1's smoke test already exercised end to end.
# screening_pooled is seeded with the Stage 3 SVM baseline and grows across
# Stages 5-7 as each pathway family screens.
#
# NOTE ON SCALE: full 28-fold LOSO x all ablation variants x (detection +
# severity) does not fit a bounded compute budget - a single
# wav2vec-fine-tuning variant's full LOSO pass alone can run tens of
# GPU-hours. So Stages 5-7 rank every variant cheaply on two axes at once:
# CV_PROTOCOL="screening" (src.splits.iter_screening_folds: SCREENING_FOLDS
# speaker-grouped folds instead of 28 single-speaker folds) AND a shorter
# SCREENING_EPOCHS/SCREENING_PATIENCE budget than the full run's
# DEFAULT_EPOCHS=20/DEFAULT_PATIENCE=5 - screening only needs to rank
# variants relative to each other, not train any one of them to full
# convergence, and only Stage 8b spends full-length full-LOSO GPU time - on
# the top TOP_K_FOR_DETECTION winners, not all six.
#
# BUDGET MANAGER: instead of hand-picking SESSION_BUDGET_HOURS and hoping it
# fits (the old pattern), src.training.budget.ExperimentBudgetManager
# benchmarks one screening fold x one epoch per variant (real wall-clock),
# projects each variant's full screening cost, and allocates a deadline per
# variant inside the hard cap - fed straight into the SAME
# run_training(..., deadline=...) call every stage already used. Every
# training cell below is still session-bounded and resumes on re-run via
# run_training()'s on-disk fold cache (src/training/runner.py) - the budget
# manager only changes how the deadline is chosen, not the fold-boundary-
# safe stopping itself.
import time

from src.training.budget import ExperimentBudgetManager
from src.training.runner import TrainingConfig, run_training

# 1.5, not 6: total wall-clock across every budgeted stage in this notebook
# (this screening pass + Stage 8b's top-2 full LOSO + Stage 8d's primary
# 6-variant sweep + Stage 10-12's severity run) is sized to land under a
# 7-hour session ceiling - screening 1.5h + Stage 8b 2.0h + Stage 8d 2.5h +
# severity 0.5h = 6.5h, leaving headroom for the one-time baseline SVM
# extraction (Stages 2-4). Screening only needs to RANK variants relative to
# each other (Stage 8d re-trains the real winners at full LOSO regardless),
# so it gets the smallest slice - raise it back toward 6 only if you have a
# larger total time budget than 7 hours for the whole notebook.
SESSION_BUDGET_HOURS = 1.5
SCREENING_FOLDS = 8         # speaker-grouped folds used to rank all variants before full LOSO
SCREENING_EPOCHS = 10       # half of DEFAULT_EPOCHS - ranking needs relative signal, not convergence
SCREENING_PATIENCE = 3      # tighter than DEFAULT_PATIENCE=5 - stop unpromising variants sooner

screening_budget = ExperimentBudgetManager(
    models=MFCC_FAMILY + WAV2VEC_FAMILY + FUSION_FAMILY,
    hard_cap_hours=SESSION_BUDGET_HOURS, n_folds=SCREENING_FOLDS,
    epochs_per_fold_estimate=SCREENING_PATIENCE + 2)
screening_budget.benchmark(df_m6, task="detection", cfg_overrides={
    "cv_protocol": "screening", "screening_folds": SCREENING_FOLDS})
screening_budget.allocate()

screening_pooled = {"baseline_svm": baseline_pooled}

for model_name in MFCC_FAMILY:
    deadline = screening_budget.deadline_for(model_name)
    if deadline is None:
        continue
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_screen_{model_name}",
        cv_protocol="screening", screening_folds=SCREENING_FOLDS,
        epochs=SCREENING_EPOCHS, patience=SCREENING_PATIENCE,
    )
    start = time.monotonic()
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    screening_budget.record_actual(model_name, time.monotonic() - start)
    if pooled:
        screening_pooled[model_name] = pooled


══════════════════════════════════════════════════════════════════════════════
  EXPERIMENT BUDGET MANAGER — BENCHMARKING
══════════════════════════════════════════════════════════════════════════════
  Frozen embedding cache .................. warming (one-time, outside the timed benchmark)


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3368.18it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



══════════════════════════════════════════════════════════════════════════════
  EXTRACTING FROZEN WAV2VEC 2.0 EMBEDDINGS (ATTENTION-MASKED)
══════════════════════════════════════════════════════════════════════════════
  Utterances .............................. 21381
  Device .................................. cuda
  Frozen wav2vec 2.0 forward (masked) 100%|██████████████████| 1337/1337 [04:13<00:00,  5.27batch/s]
  Frozen embeddings (masked) .............. extracted and cached to C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\embeddings\frozen_wav2vec_base_masked.npz

╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│               Model A — MFCC 1D-CNN (cepstral features only)               │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3861.72it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: screen1  │  train 16,502 / val 1,834 / test 3,045
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_frozen ───────────────────────────────────────────────
    deep_pathway ..........................           0 /  94,371,712 trainable (  0.0%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 197,378 / 94,569,090 (0.21%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/1  │  train  loss 0.6052  acc 0.664  │  val  loss 0.5686  acc 0.694  f1 0.675   <-- best
  Fold screen1 held-out test .............. accuracy=0.526, precision=0.539, recall=0.327, specificity=0.723, f1=0.407, auroc=0.518

══════════════════════════════════════════════════════════════════════════════
  RESULTS — 

...


──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: screen1  │  train 16,502 / val 1,834 / test 3,045
──────────────────────────────────────────────────────────────────────────────

─── Architecture — fusion ────────────────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    acoustic_pathway ......................     103,552 /     103,552 trainable (100.0%)
    classifier ............................     230,146 /     230,146 trainable (100.0%)
  TOTAL trainable ......................... 776,066 / 95,147,778 (0.82%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/1  │  train  loss 0.2570  acc 0.890  │  val  loss 0.1937  acc 0.936  f1 0.943   <-- best
  Fold screen1 held-out test .............. accuracy=0.743, precision=0.670, recall=0.949, specificity=0.538, f1=0.786, auroc=0.863

══

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 4580.96it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: screen1  │  train 16,502 / val 1,834 / test 3,045
──────────────────────────────────────────────────────────────────────────────

─── Architecture — attention_fusion ──────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    acoustic_pathway ......................     103,552 /     103,552 trainable (100.0%)
    deep_proj .............................     196,864 /     196,864 trainable (100.0%)
    acoustic_proj .........................      33,024 /      33,024 trainable (100.0%)
    deep_from_acoustic ....................     527,616 /     527,616 trainable (100.0%)
    acoustic_from_deep ....................     527,616 /     527,616 trainable (100.0%)
    classifier ............................     131,842 /     131,842 trainable (100.0%)
  Fused embedding ......................... 512-dim
  T

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3036.46it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: screen1  │  train 16,502 / val 1,834 / test 3,045
──────────────────────────────────────────────────────────────────────────────

─── Architecture — attention_fusion_praat ────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    acoustic_pathway ......................     103,552 /     103,552 trainable (100.0%)
    deep_proj .............................     196,864 /     196,864 trainable (100.0%)
    acoustic_proj .........................      33,024 /      33,024 trainable (100.0%)
    praat_encoder .........................      74,496 /      74,496 trainable (100.0%)
    deep_from_context .....................     527,616 /     527,616 trainable (100.0%)
    acoustic_from_context .................     527,616 /     527,616 trainable (100.0%)
    classifier ............................     197,378

In [8]:
# STAGE 6 - Wav2Vec2-only screening (detection). Frozen and LoRA-adapted
# wav2vec 2.0 + MLP head (Models B and C), on the same cheap screening
# protocol and shortened epoch budget as Stage 5. Continues screening_pooled
# and reuses the SAME screening_budget from Stage 5 (already benchmarked
# every variant, including this family) - same session-bounded,
# resume-on-rerun pattern.
from src.training.runner import TrainingConfig, run_training

for model_name in WAV2VEC_FAMILY:
    deadline = screening_budget.deadline_for(model_name)
    if deadline is None:
        continue
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_screen_{model_name}",
        cv_protocol="screening", screening_folds=SCREENING_FOLDS,
        epochs=SCREENING_EPOCHS, patience=SCREENING_PATIENCE,
    )
    start = time.monotonic()
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    screening_budget.record_actual(model_name, time.monotonic() - start)
    if pooled:
        screening_pooled[model_name] = pooled

  Frozen embeddings (masked) .............. loaded from cache (C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\embeddings\frozen_wav2vec_base_masked.npz)

╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│                  Model B — frozen wav2vec 2.0 + MLP head                   │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ................................... deep_frozen — Model B — frozen wav2vec 2.0 + MLP head
  Cross-validation protocol ............... screening (8-fold, speaker-grouped)
  Run name ................................ detection_screen_deep_frozen
  Device .................................. cuda
  Epochs / batch size ......

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 2886.41it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/8  │  held-out speaker: screen1  │  train 16,502 / val 1,834 / test 3,045
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_frozen ───────────────────────────────────────────────
    deep_pathway ..........................           0 /  94,371,712 trainable (  0.0%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 197,378 / 94,569,090 (0.21%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/10  │  train  loss 0.6052  acc 0.664  │  val  loss 0.5686  acc 0.694  f1 0.675   <-- best
    epoch   2/10  │  train  loss 0.5633  acc 0.701  │  val  loss 0.5411  acc 0.718  f1 0.727   <-- best
    epoch   3/10  │  train  loss 0.5477  acc 0.711  │  val  loss 0.5525  acc 0.703  f1 0.652        
    epoch   4/10  

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3867.62it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/8  │  held-out speaker: screen1  │  train 16,502 / val 1,834 / test 3,045
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_lora ─────────────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 639,746 / 95,011,458 (0.67%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/10  │  train  loss 0.5880  acc 0.656  │  val  loss 0.4346  acc 0.805  f1 0.806   <-- best
    epoch   2/10  │  train  loss 0.4289  acc 0.799  │  val  loss 0.3761  acc 0.832  f1 0.820   <-- best
    epoch   3/10  │  train  loss 0.3490  acc 0.849  │  val  loss 0.3231  acc 0.865  f1 0.860   <-- best
    epoch   4/1

In [9]:
# STAGE 7 - Fusion model screening (detection). Frozen-backbone and
# LoRA-backbone concatenated Fusion (Model D and its frozen counterpart) and
# the two Phase 6 attention-fusion variants (Models E, F), on the same cheap
# screening protocol and shortened epoch budget as Stages 5-6. Continues
# screening_pooled and reuses screening_budget from Stage 5 - same
# session-bounded, resume-on-rerun pattern.
from src.training.runner import TrainingConfig, run_training

for model_name in FUSION_FAMILY:
    deadline = screening_budget.deadline_for(model_name)
    if deadline is None:
        continue
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_screen_{model_name}",
        cv_protocol="screening", screening_folds=SCREENING_FOLDS,
        epochs=SCREENING_EPOCHS, patience=SCREENING_PATIENCE,
    )
    start = time.monotonic()
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    screening_budget.record_actual(model_name, time.monotonic() - start)
    if pooled:
        screening_pooled[model_name] = pooled

  • Session budget (1.5h) already used up — skipping 'fusion_frozen'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).
  • Session budget (1.5h) already used up — skipping 'fusion'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).
  • Session budget (1.5h) already used up — skipping 'attention_fusion'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).
  • Session budget (1.5h) already used up — skipping 'attention_fusion_praat'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).


In [10]:
# STAGE 8 - Screening comparison table: baseline SVM vs. every screened
# variant from Stages 5-7, pooled metrics side by side. This ranks the six
# ablation variants cheaply; it is NOT the reportable LOSO result (Stage 8b
# runs that for the winners only). Saved to
# outputs/metrics/phase2_screening_comparison.csv.
from src.model_analysis import plot_ablation_comparison
from src.results import style_comparison_table

screening_df = pd.DataFrame(screening_pooled).T
screening_df.index.name = "model"

screening_path = config.METRICS_DIR / "phase2_screening_comparison.csv"
screening_df.to_csv(screening_path)

print_header("Phase 2/3 Screening Comparison - Detection")
print_kv("Saved to", screening_path)

loss_cols = [c for c in screening_df.columns if "loss" in c]
score_cols = [c for c in screening_df.columns if c not in loss_cols]
display(style_comparison_table(screening_df[score_cols]))
if loss_cols:
    display(style_comparison_table(screening_df[loss_cols], higher_is_better=False))

plot_ablation_comparison(screening_df, title="Ablation Screening - Detection", show=True)


══════════════════════════════════════════════════════════════════════════════
  PHASE 2/3 SCREENING COMPARISON - DETECTION
══════════════════════════════════════════════════════════════════════════════
  Saved to ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\phase2_screening_comparison.csv


,accuracy,precision,recall,specificity,f1,auroc
model,,,,,,
baseline_svm,0.8225,0.8796,0.7741,0.8781,0.8235,0.8862
acoustic,0.8128,0.7714,0.8865,0.7399,0.8249,0.9073
deep_frozen,0.5488,0.5592,0.4396,0.6569,0.4922,0.5763
deep_lora,0.8539,0.8132,0.9168,0.7915,0.8619,0.9482


c:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\src\model_analysis.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'C:\\Users\\surya\\OneDrive\\Desktop\\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\\outputs\\figures\\ablation_comparison.png'

In [11]:
# STAGE 8b - Full 28-fold LOSO training (detection), winners only. This is
# where the compute budget actually gets spent deliberately: instead of
# full LOSO x all 6 variants, only the top TOP_K_FOR_DETECTION variants from
# Stage 8's cheap screening get the real base-paper protocol. That is what
# makes comparison_pooled below the number worth reporting/citing - the
# screening_pooled numbers above rank variants but are not full LOSO.
# Same session-bounded, resume-on-rerun pattern as Stages 5-7.
from src.training.runner import TrainingConfig, run_training
from src.training.models import MODEL_NAMES
from src.console import print_note
import time

TOP_K_FOR_DETECTION = 2

ranked_by_screening = sorted(
    (m for m in MODEL_NAMES if m in screening_pooled),
    key=lambda m: screening_pooled[m]["f1"], reverse=True,
)
detection_loso_candidates = ranked_by_screening[:TOP_K_FOR_DETECTION]
print_note(f"Running full 28-fold LOSO only for top {TOP_K_FOR_DETECTION} "
          f"screened variants by F1: {detection_loso_candidates}")

# 2.0, not 6: see Stage 5's budget comment - this is one of four budgeted
# stages sized to sum to <=7h total (screening 1.5h + this 2.0h + Stage 8d
# 2.5h + severity 0.5h = 6.5h).
SESSION_BUDGET_HOURS = 2.0
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

comparison_pooled = {"baseline_svm": baseline_pooled}

for model_name in detection_loso_candidates:
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"detection_{model_name}",
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        comparison_pooled[model_name] = pooled

  • Running full 28-fold LOSO only for top 2 screened variants by F1: ['deep_lora', 'acoustic']

╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│              Model C — wav2vec 2.0 + LoRA adapters + MLP head              │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ................................... deep_lora — Model C — wav2vec 2.0 + LoRA adapters + MLP head
  Cross-validation protocol ............... Leave-One-Speaker-Out
  Run name ................................ detection_deep_lora
  Device .................................. cuda
  Epochs / batch size ..................... 20 / 32
  LR (head / wav2vec backbone) ............ 0.001 / 0.0001
  Early stopping patience ................. 5 ep

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 4431.90it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/28  │  held-out speaker: CF02  │  train 18,554 / val 2,062 / test 765
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_lora ─────────────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 639,746 / 95,011,458 (0.67%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/20  │  train  loss 0.5898  acc 0.650  │  val  loss 0.4434  acc 0.809  f1 0.812   <-- best
    epoch   2/20  │  train  loss 0.4344  acc 0.795  │  val  loss 0.3463  acc 0.852  f1 0.853   <-- best
    epoch   3/20  │  train  loss 0.3548  acc 0.846  │  val  loss 0.2811  acc 0.887  f1 0.889   <-- best
    epoch   4/20  │

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3770.68it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 2/28  │  held-out speaker: CF03  │  train 18,554 / val 2,062 / test 765
──────────────────────────────────────────────────────────────────────────────
    epoch   1/20  │  train  loss 0.6068  acc 0.635  │  val  loss 0.6873  acc 0.777  f1 0.797   <-- best
    epoch   2/20  │  train  loss 0.4594  acc 0.784  │  val  loss 0.3400  acc 0.862  f1 0.862   <-- best
    epoch   3/20  │  train  loss 0.3608  acc 0.845  │  val  loss 0.3866  acc 0.885  f1 0.896        
    epoch   4/20  │  train  loss 0.2938  acc 0.881  │  val  loss 0.2112  acc 0.927  f1 0.933   <-- best
    epoch   5/20  │  train  loss 0.2500  acc 0.902  │  val  loss 0.1695  acc 0.940  f1 0.947   <-- best
    epoch   6/20  │  train  loss 0.2137  acc 0.918  │  val  loss 0.1010  acc 0.958  f1 0.962   <-- best
    epoch   7/20  │  train  loss 0.1856  acc 0.931  │  val  loss 0.2614  acc 0.939  f1 0.947        
    epoch   8/20  │  train  loss 0.1834 

In [12]:
# STAGE 8c - Phase 2/3 comparison table: baseline SVM vs. Stage 8b's full-LOSO
# winners, pooled metrics side by side. This is the reportable, base-paper-
# comparable detection result (unlike Stage 8's screening table). Saved to
# outputs/metrics/phase2_comparison.csv for the paper/report.
from src.model_analysis import plot_ablation_comparison
from src.results import style_comparison_table

comparison_df = pd.DataFrame(comparison_pooled).T
comparison_df.index.name = "model"

comparison_path = config.METRICS_DIR / "phase2_comparison.csv"
comparison_df.to_csv(comparison_path)

print_header("Phase 2/3 Comparison - Detection (full LOSO)")
print_kv("Saved to", comparison_path)

loss_cols = [c for c in comparison_df.columns if "loss" in c]
score_cols = [c for c in comparison_df.columns if c not in loss_cols]
display(style_comparison_table(comparison_df[score_cols]))
if loss_cols:
    display(style_comparison_table(comparison_df[loss_cols], higher_is_better=False))

plot_ablation_comparison(comparison_df, title="Ablation Comparison - Detection (Full LOSO)", show=True)


══════════════════════════════════════════════════════════════════════════════
  PHASE 2/3 COMPARISON - DETECTION (FULL LOSO)
══════════════════════════════════════════════════════════════════════════════
  Saved to ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\phase2_comparison.csv


,accuracy,precision,recall,specificity,f1,auroc
model,,,,,,
baseline_svm,0.8225,0.8796,0.7741,0.8781,0.8235,0.8862
deep_lora,0.9974,0.0000,0.0000,0.9974,0.0000,nan


c:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\src\model_analysis.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'C:\\Users\\surya\\OneDrive\\Desktop\\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\\outputs\\figures\\ablation_comparison.png'

In [13]:
# STAGE 8d - PRIMARY DETECTION SWEEP (supervisor-requested): the six
# necessary variants (PRIMARY_DETECTION_MODELS, Experiment Manager cell)
# under a measured, explicit compute budget - MFCC baseline, frozen
# Wav2Vec2 baseline, LoRA-Wav2Vec2, MFCC+frozen-Wav2Vec2, MFCC+LoRA-Wav2Vec2
# ("LoRA-Fusion"), and the proposed attention-fusion architecture. Unlike
# Stages 5-8c (screening then top-K full LOSO), every one of these six runs
# the FULL 28-fold LOSO protocol - this is the reportable comparison table
# (Stage 8e) requirement 7 asks for, and requirement 2's "does LoRA help
# Wav2Vec2? does LoRA help fusion?" questions are answered by reading
# deep_frozen-vs-deep_lora and fusion_frozen-vs-fusion pairs straight off it,
# not by a separate duplicated run.
#
# ExperimentBudgetManager.benchmark() measures one real LOSO fold x one real
# epoch per variant (not a guess), projects each variant's full-28-fold cost,
# and allocates a deadline per variant inside the hard cap -
# proportional to measured cost, so the MFCC CNN doesn't get the same
# wall-clock slice as attention_fusion. Each variant's outputs also get
# repackaged into outputs/experiments/<name>/ (config.json, metrics.json,
# predictions.csv, timing.json, checkpoint/) via save_experiment_bundle -
# additive to the flat outputs/<kind>/<run_name>/ layout run_training()
# already writes.
import time

from src.training.budget import ExperimentBudgetManager
from src.training.reporting import save_experiment_bundle
from src.training.runner import TrainingConfig, run_training

# 2.5, not 6: see Stage 5's budget comment - this is the largest slice of
# the <=7h total session budget (screening 1.5h + Stage 8b 2.0h + this 2.5h
# + severity 0.5h = 6.5h) since it produces the primary reportable result.
PRIMARY_HARD_CAP_HOURS = 2.5

primary_budget = ExperimentBudgetManager(
    models=PRIMARY_DETECTION_MODELS, hard_cap_hours=PRIMARY_HARD_CAP_HOURS, n_folds=28)
primary_budget.benchmark(df_m6, task="detection")
primary_budget.allocate()

primary_pooled = {}
primary_summaries = {}
primary_bundle_paths = {}

for model_name in PRIMARY_DETECTION_MODELS:
    deadline = primary_budget.deadline_for(model_name)
    if deadline is None:
        continue
    cfg = TrainingConfig(
        task="detection", model=model_name,
        run_name=f"primary_detection_{model_name}",
        cv_protocol="loso",
    )
    start = time.monotonic()
    summary, pooled = run_training(df_m6, cfg, deadline=deadline)
    primary_budget.record_actual(model_name, time.monotonic() - start)
    if not pooled:
        continue
    primary_pooled[model_name] = pooled
    primary_summaries[model_name] = summary
    primary_bundle_paths[model_name] = save_experiment_bundle(
        experiment_name=model_name, model_name=model_name, task="detection",
        cfg=cfg, pooled_metrics=pooled, summary=summary,
        num_classes=config.NUM_CLASSES["detection"])

print_header("Primary Detection Sweep — bundles saved")
for model_name, path in primary_bundle_paths.items():
    print_kv(model_name, path)


══════════════════════════════════════════════════════════════════════════════
  EXPERIMENT BUDGET MANAGER — BENCHMARKING
══════════════════════════════════════════════════════════════════════════════
  Frozen embedding cache .................. warming (one-time, outside the timed benchmark)
  Frozen embeddings (masked) .............. loaded from cache (C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\embeddings\frozen_wav2vec_base_masked.npz)

╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│               Model A — MFCC 1D-CNN (cepstral features only)               │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ..........................

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 4711.11it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: CF02  │  train 18,554 / val 2,062 / test 765
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_frozen ───────────────────────────────────────────────
    deep_pathway ..........................           0 /  94,371,712 trainable (  0.0%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 197,378 / 94,569,090 (0.21%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/1  │  train  loss 0.6121  acc 0.651  │  val  loss 0.5867  acc 0.687  f1 0.714   <-- best
  Fold CF02 held-out test ................. accuracy=0.684, precision=0.000, recall=0.000, specificity=0.684, f1=0.000, auroc=nan

══════════════════════════════════════════════════════════════════════════════
  RESULTS — _BUDGET

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 4618.21it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/1  │  held-out speaker: CF02  │  train 18,554 / val 2,062 / test 765
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_lora ─────────────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 639,746 / 95,011,458 (0.67%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/1  │  train  loss 0.5796  acc 0.663  │  val  loss 0.5050  acc 0.745  f1 0.708   <-- best
  Fold CF02 held-out test ................. accuracy=0.978, precision=0.000, recall=0.000, specificity=0.978, f1=0.000, auroc=nan

══════════════════════════════════════════════════════════════════════════════
  RESULTS — _BUDGET

...


╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│              Model C — wav2vec 2.0 + LoRA adapters + MLP head              │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... detection (2-class)
  Model ................................... deep_lora — Model C — wav2vec 2.0 + LoRA adapters + MLP head
  Cross-validation protocol ............... Leave-One-Speaker-Out
  Run name ................................ primary_detection_deep_lora
  Device .................................. cuda
  Epochs / batch size ..................... 20 / 32
  LR (head / wav2vec backbone) ............ 0.001 / 0.0001
  Early stopping patience ................. 5 epochs on validation loss

─── Front end — short-time analysis ───────────────────────────

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 7020.59it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/28  │  held-out speaker: CF02  │  train 18,554 / val 2,062 / test 765
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_lora ─────────────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    classifier ............................     197,378 /     197,378 trainable (100.0%)
  TOTAL trainable ......................... 639,746 / 95,011,458 (0.67%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/20  │  train  loss 0.6052  acc 0.637  │  val  loss 0.5379  acc 0.767  f1 0.763   <-- best
    epoch   2/20  │  train  loss 0.4468  acc 0.786  │  val  loss 0.3879  acc 0.809  f1 0.793   <-- best
    epoch   3/20  │  train  loss 0.3641  acc 0.838  │  val  loss 0.3305  acc 0.872  f1 0.872   <-- best
    epoch   4/20  │

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 3435.35it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  • Session budget (2.5h) already used up — skipping 'fusion_frozen'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).
  • Session budget (2.5h) already used up — skipping 'fusion'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).
  • Session budget (2.5h) already used up — skipping 'attention_fusion'. Re-run this cell later to resume (completed folds are loaded from disk, not retrained).

══════════════════════════════════════════════════════════════════════════════
  PRIMARY DETECTION SWEEP — BUNDLES SAVED
══════════════════════════════════════════════════════════════════════════════
  acoustic ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\experiments\acoustic
  deep_frozen ............................. C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\experiment

In [14]:
# STAGE 8e - Final comparison table (requirement 7): Model | Input |
# Adaptation | Trainable Params | Accuracy | Precision | Recall | F1 | AUC |
# Training Time, for every variant Stage 8d completed. Reuses
# src.results.style_comparison_table (same colour-graded renderer as Stages
# 8/8c/13) rather than a new one. Saved to
# outputs/metrics/primary_detection_comparison.csv, matching the
# outputs/experiments/<name>/metrics.json bundles Stage 8d already wrote.
from src.console import print_table
from src.results import style_comparison_table
from src.training.models import build_model, parameter_counts

VARIANT_INFO = {
    "acoustic":          {"Input": "MFCC",                          "Adaptation": "—"},
    "deep_frozen":       {"Input": "Wav2Vec2",                      "Adaptation": "Frozen"},
    "deep_lora":         {"Input": "Wav2Vec2",                      "Adaptation": "LoRA"},
    "fusion_frozen":     {"Input": "MFCC + Wav2Vec2",                "Adaptation": "Frozen"},
    "fusion":            {"Input": "MFCC + Wav2Vec2",                "Adaptation": "LoRA"},
    "attention_fusion":  {"Input": "MFCC + Wav2Vec2 (cross-attn)",   "Adaptation": "LoRA"},
}

rows = []
for model_name, pooled in primary_pooled.items():
    params = parameter_counts(build_model(model_name, config.NUM_CLASSES["detection"]))
    train_time_h = (primary_summaries[model_name].loc["train_time_s", "mean"] * 28 / 3600
                    if "train_time_s" in primary_summaries[model_name].index else float("nan"))
    rows.append({
        "Model": model_name, **VARIANT_INFO.get(model_name, {}),
        "Trainable Params": params["trainable_params"],
        "Accuracy": pooled["accuracy"], "Precision": pooled["precision"],
        "Recall": pooled["recall"], "F1": pooled["f1"], "AUC": pooled["auroc"],
        "Training Time (h)": train_time_h,
    })

primary_comparison_df = pd.DataFrame(rows).set_index("Model")
primary_comparison_path = config.METRICS_DIR / "primary_detection_comparison.csv"
primary_comparison_df.to_csv(primary_comparison_path)

print_header("Primary Detection Sweep — Final Comparison")
print_kv("Saved to", primary_comparison_path)

info_cols = ["Input", "Adaptation", "Trainable Params", "Training Time (h)"]
score_cols = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
print_table(primary_comparison_df[info_cols].reset_index())
display(style_comparison_table(primary_comparison_df[score_cols]))

# ----------------------------------------------------------------------
# LoRA ablation read-off (requirement 2): no new training - two deltas read
# straight off the table above.
#   A. LoRA-Wav2Vec:  deep_frozen  -> deep_lora        (does LoRA beat frozen wav2vec2?)
#   B. LoRA-Fusion:   fusion_frozen -> fusion           (does LoRA help the fusion architecture?)
lora_pairs = [("deep_frozen", "deep_lora"), ("fusion_frozen", "fusion")]
lora_rows = []
for frozen_name, lora_name in lora_pairs:
    if frozen_name not in primary_comparison_df.index or lora_name not in primary_comparison_df.index:
        continue
    frozen_row = primary_comparison_df.loc[frozen_name]
    lora_row = primary_comparison_df.loc[lora_name]
    lora_rows.append({
        "Comparison": f"{frozen_name} → {lora_name}",
        "Trainable Params (frozen → LoRA)":
            f"{frozen_row['Trainable Params']:,.0f} → {lora_row['Trainable Params']:,.0f}",
        "Accuracy delta": lora_row["Accuracy"] - frozen_row["Accuracy"],
        "F1 delta": lora_row["F1"] - frozen_row["F1"],
        "AUC delta": lora_row["AUC"] - frozen_row["AUC"],
    })

lora_delta_df = pd.DataFrame(lora_rows).set_index("Comparison")
lora_delta_path = config.METRICS_DIR / "lora_ablation_deltas.csv"
lora_delta_df.to_csv(lora_delta_path)

print_header("LoRA Ablation — Does LoRA Help?")
print_kv("Saved to", lora_delta_path)
print_table(lora_delta_df.reset_index())

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 6849.55it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 210/210 [00:00<00:00, 7694.89it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newl


══════════════════════════════════════════════════════════════════════════════
  PRIMARY DETECTION SWEEP — FINAL COMPARISON
══════════════════════════════════════════════════════════════════════════════
  Saved to ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\primary_detection_comparison.csv
        Model    Input Adaptation  Trainable Params  Training Time (h)
     acoustic     MFCC          —            111938            15.9328
  deep_frozen Wav2Vec2     Frozen            197378            17.8628
    deep_lora Wav2Vec2       LoRA            639746            37.7710


c:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\surya\anaconda3\envs\torch-gpu\lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax


,Accuracy,Precision,Recall,F1,AUC
Model,,,,,
acoustic,0.9673,0.0000,0.0000,0.0000,nan
deep_frozen,0.8196,0.0000,0.0000,0.0000,nan
deep_lora,0.9974,0.0000,0.0000,0.0000,nan



══════════════════════════════════════════════════════════════════════════════
  LORA ABLATION — DOES LORA HELP?
══════════════════════════════════════════════════════════════════════════════
  Saved to ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\lora_ablation_deltas.csv
               Comparison Trainable Params (frozen → LoRA)  Accuracy delta  F1 delta  AUC delta
  deep_frozen → deep_lora                197,378 → 639,746          0.1778    0.0000        NaN


In [15]:
# STAGE 9 - Pick which variant(s) go on to the severity task. Stages 10-12
# below run the severity protocol (leave-one-speaker-per-class-out,
# subsampled to SEVERITY_FOLD_SAMPLE of the base paper's 81 combinations -
# see Stage 10), the single largest remaining GPU-time item in this
# notebook. Only the architecture that actually won Stage 8c's full-LOSO
# detection comparison matters for the severity story, so rank those
# results by F1 and carry forward just the top TOP_K_FOR_SEVERITY (picked
# from detection_loso_candidates, not all six variants - the others never
# got a full-LOSO score to rank by).
TOP_K_FOR_SEVERITY = 1

ranked_variants = sorted(
    (m for m in detection_loso_candidates if m in comparison_pooled),
    key=lambda m: comparison_pooled[m]["f1"], reverse=True,
)
severity_model_names = ranked_variants[:TOP_K_FOR_SEVERITY]

print_note(f"Running severity only for top {TOP_K_FOR_SEVERITY} "
          f"full-LOSO detection variant(s) by F1: {severity_model_names}")

  • Running severity only for top 1 full-LOSO detection variant(s) by F1: ['deep_lora']


In [16]:
# STAGE 10 - Severity training - MFCC-only. Runs src.training.runner over
# MFCC_FAMILY, restricted to whichever of those variants made Stage 9's
# top-TOP_K_FOR_SEVERITY cut; if none did, the loop below is simply a no-op.
# Balanced leave-one-speaker-per-class-out protocol (src/splits.py,
# config.DROPPED_FOR_BALANCE), subsampled to SEVERITY_FOLD_SAMPLE of the
# base paper's 81 combinations (src.splits.sample_severity_folds) - 81 folds
# is a bigger job than detection's 28-fold LOSO even before Stage 9 already
# cut this down to one variant, and it's still the largest remaining
# GPU-time item in a bounded total-compute budget. Plus the Stage 4 severity
# SVM baseline for a like-for-like comparison table (mirrors Stages 5-8's
# detection pattern). Same session-bounded, resume-on-rerun pattern as
# Stages 5-7.
#
# Lower priority than Stages 5-8c's detection run (the paper's primary
# comparison) - run Stages 10-12 once detection is done or far enough along.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

# 0.5, not 6: see Stage 5's budget comment - severity gets the smallest
# slice of the <=7h total session budget (screening 1.5h + Stage 8b 2.0h +
# Stage 8d 2.5h + this 0.5h = 6.5h). It only trains ONE variant (Stage 9's
# top pick) on a 20-of-81-fold subsample, so it needs far less wall-clock
# than the detection stages to begin with.
SESSION_BUDGET_HOURS = 0.5
SEVERITY_FOLD_SAMPLE = 20   # of the base paper's 81 leave-one-per-class-out combinations
deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

severity_pooled = {"baseline_svm": severity_baseline_pooled}

for model_name in (m for m in MFCC_FAMILY if m in severity_model_names):
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
        severity_fold_sample=SEVERITY_FOLD_SAMPLE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

In [17]:
# STAGE 11 - Severity training - Wav2Vec2-only. Continues severity_pooled
# from Stage 10, restricted to whichever of WAV2VEC_FAMILY made the Stage 9
# cut. Same SEVERITY_FOLD_SAMPLE subsample as Stage 10.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

for model_name in (m for m in WAV2VEC_FAMILY if m in severity_model_names):
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
        severity_fold_sample=SEVERITY_FOLD_SAMPLE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled


╔════════════════════════════════════════════════════════════════════════════╗
│                    UA-SPEECH DYSARTHRIA CLASSIFICATION                     │
│              Model C — wav2vec 2.0 + LoRA adapters + MLP head              │
╚════════════════════════════════════════════════════════════════════════════╝

─── Run configuration ────────────────────────────────────────────────────────
  Task .................................... severity (4-class)
  Model ................................... deep_lora — Model C — wav2vec 2.0 + LoRA adapters + MLP head
  Cross-validation protocol ............... balanced leave-one-speaker-per-class-out (subsampled to 20 of 81)
  Run name ................................ severity_deep_lora
  Device .................................. cuda
  Epochs / batch size ..................... 20 / 32
  LR (head / wav2vec backbone) ............ 0.001 / 0.0001
  Early stopping patience ................. 5 epochs on validation loss

─── Front end — short-time an

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 6161.28it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



──────────────────────────────────────────────────────────────────────────────
  FOLD 1/20  │  held-out speaker: F05-M07-M05-M04  │  train 5,472 / val 609 / test 3,060
──────────────────────────────────────────────────────────────────────────────

─── Architecture — deep_lora ─────────────────────────────────────────────────
    deep_pathway ..........................     442,368 /  94,814,080 trainable (  0.5%)
    classifier ............................     197,892 /     197,892 trainable (100.0%)
  TOTAL trainable ......................... 640,260 / 95,011,972 (0.67%)
  • The frozen remainder is wav2vec 2.0's pre-trained backbone — only the adapters and head learn.

    epoch   1/20  │  train  loss 1.3130  acc 0.346  │  val  loss 1.2699  acc 0.414  f1 0.350   <-- best
    epoch   2/20  │  train  loss 1.1452  acc 0.463  │  val  loss 0.9980  acc 0.544  f1 0.488   <-- best
    epoch   3/20  │  train  loss 0.9814  acc 0.562  │  val  loss 0.9489  acc 0.550  f1 0.521   <-- best
    epoch

In [18]:
# STAGE 12 - Severity training - Fusion. Continues severity_pooled from
# Stages 10-11, restricted to whichever of FUSION_FAMILY made the Stage 9
# cut. Same SEVERITY_FOLD_SAMPLE subsample as Stage 10.
from src.training.runner import TrainingConfig, run_training
from src.console import print_note
import time

deadline = time.monotonic() + SESSION_BUDGET_HOURS * 3600

for model_name in (m for m in FUSION_FAMILY if m in severity_model_names):
    if time.monotonic() >= deadline:
        print_note(f"Session budget used up before starting '{model_name}' - "
                   "re-run this cell later to continue.")
        break
    cfg = TrainingConfig(
        task="severity", model=model_name,
        run_name=f"severity_{model_name}",
        severity_fold_sample=SEVERITY_FOLD_SAMPLE,
    )
    _, pooled = run_training(df_m6, cfg, deadline=deadline)
    if pooled:
        severity_pooled[model_name] = pooled

In [19]:
# STAGE 13 - Phase 3 severity comparison table: baseline SVM vs. Stages
# 10-12's severity-trained variant(s) (subsampled folds - see Stage 10),
# pooled metrics side by side. Saved to
# outputs/metrics/phase3_severity_comparison.csv for the paper/report. Same
# colour-graded display as Stage 8c.
from src.results import style_comparison_table

severity_df = pd.DataFrame(severity_pooled).T
severity_df.index.name = "model"

severity_path = config.METRICS_DIR / "phase3_severity_comparison.csv"
severity_df.to_csv(severity_path)

print_header("Phase 3 Comparison - Severity")
print_kv("Saved to", severity_path)

loss_cols = [c for c in severity_df.columns if "loss" in c]
score_cols = [c for c in severity_df.columns if c not in loss_cols]
display(style_comparison_table(severity_df[score_cols]))
if loss_cols:
    display(style_comparison_table(severity_df[loss_cols], higher_is_better=False))


══════════════════════════════════════════════════════════════════════════════
  PHASE 3 COMPARISON - SEVERITY
══════════════════════════════════════════════════════════════════════════════
  Saved to ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\phase3_severity_comparison.csv


,accuracy,precision,recall,specificity,f1,auroc
model,,,,,,
baseline_svm,0.4779,0.4582,0.4778,0.8260,0.4662,0.6826
deep_lora,0.5676,0.6174,0.5676,0.8559,0.5755,0.8320
